# Gemini 이미지 캡셔닝 최적화 실험

**목적**: 이미지 전달 방식 및 이미지 크기에 따른 성능·비용 비교

---

## 실험 구성

| Phase | 내용 | 핵심 질문 |
|-------|------|-----------|
| 1 | URL 접근 가능성 검증 | 어떤 URL 형식이 Gemini에서 직접 동작하는가? |
| 2 | 전달 방식별 속도 비교 | base64(현재) vs GCS URI 중 어느 것이 빠른가? |
| 3 | `media_resolution` 토큰 절감 | LOW/MEDIUM/HIGH 중 캡셔닝 품질을 유지하면서 토큰을 아낄 수 있는가? |
| 4 | 이미지 리사이즈 최적화 | 384px 이하로 줄이면 토큰 수가 실제로 줄어드는가? |

---

### Phase 1 결과 요약 (사전 실험 확정)

| 방식 | 결과 |
|------|------|
| 외부 CDN HTTP URL (한섬, 쿠팡 등) | 간헐적 성공 — 운영 불가 |
| GCS CDN HTTP URL (`http://IP`) | 실패 — IP 주소 fetch 차단 |
| GCS URI (`gs://`) | **안정적 성공** — 유일하게 신뢰 가능 |

> **결론**: URL 직접 전달은 `gs://`만 실제 운영 가능.  
> **Phase 2 비교 대상**: base64(현재 방식) vs GCS URI

---
## 0. 환경 설정

In [1]:
# 필요 패키지 설치
# !pip install google-genai google-cloud-storage requests pillow

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# ============================================================
#  설정값
# ============================================================

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
LOCATION   = os.getenv("GCP_LOCATION", "us-central1")
MODEL      = "gemini-2.5-flash-lite"
GCS_BUCKET = os.getenv("GCS_BUCKET")

# --- 테스트 이미지 URL ---
# DOWNLOAD_URL : 로컬에서 다운로드 가능한 URL (base64 방식 소스, Phase 4 소스)
# GCS_URI      : 동일 이미지의 gs:// URI (URL 직접 전달 방식)
DOWNLOAD_URL = os.getenv("TEST_DOWNLOAD_URL")
GCS_URI      = os.getenv("TEST_GCS_URI")
# ---------------------------------------------------------

CAPTION_PROMPT = "이 상품 이미지를 보고 마케팅용 한국어 캡션을 2문장으로 작성해주세요."

print(f"모델    : {MODEL}")
print(f"프로젝트: {PROJECT_ID} / 리전: {LOCATION}")
print()
print("URL 설정 상태:")
for name, val in [("DOWNLOAD_URL", DOWNLOAD_URL), ("GCS_URI", GCS_URI)]:
    print(f"  {name:13s}: {'설정됨' if val else '미설정'}")

In [3]:
import time
import io
import mimetypes
from typing import Optional

import requests
from PIL import Image
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print("google-genai 클라이언트 초기화 완료")

google-genai 클라이언트 초기화 완료


In [4]:
def detect_mime(url: str) -> str:
    mime, _ = mimetypes.guess_type(url)
    return mime or "image/jpeg"


def download_image(url: str, timeout: int = 15) -> bytes:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
    resp.raise_for_status()
    return resp.content


def call_gemini(
    part: types.Part,
    prompt: str = CAPTION_PROMPT,
    media_resolution: Optional[str] = None,
) -> dict:
    """media_resolution: 'MEDIA_RESOLUTION_LOW' | 'MEDIA_RESOLUTION_MEDIUM' | 'MEDIA_RESOLUTION_HIGH'"""
    config_kwargs: dict = {"temperature": 0}
    if media_resolution:
        config_kwargs["media_resolution"] = media_resolution

    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=MODEL,
        contents=[part, types.Part.from_text(text=prompt)],
        config=types.GenerateContentConfig(**config_kwargs),
    )
    elapsed = time.perf_counter() - t0

    usage = resp.usage_metadata
    return {
        "text":          resp.text,
        "elapsed_sec":   round(elapsed, 3),
        "input_tokens":  usage.prompt_token_count,
        "output_tokens": usage.candidates_token_count,
        "total_tokens":  usage.total_token_count,
    }


def print_result(label: str, r: dict):
    print(f"\n{'─'*50}")
    print(f"[{label}]")
    print(f"  응답 시간  : {r['elapsed_sec']}s")
    print(f"  입력 토큰  : {r['input_tokens']}")
    print(f"  출력 토큰  : {r['output_tokens']}")
    print(f"  캡션       : {r['text'][:150]}")


print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


---
## Phase 1 — URL 접근 가능성 검증 (참고용)

이미 결론이 나왔지만, 실제 환경에서 재현 확인용으로 남겨둔다.

| 방식 | 확정 결과 |
|------|----------|
| 외부 CDN HTTP URL | 간헐적 성공, 운영 불가 |
| GCS CDN HTTP URL (IP) | 실패 |
| GCS URI `gs://` | **성공** |

In [ ]:
# Phase 1: GCS URI 접근 확인 (가장 중요한 방식만 검증)
if not GCS_URI:
    print("GCS_URI 미설정 — 건너뜀")
else:
    print(f"GCS URI: {GCS_URI}")
    try:
        part   = types.Part.from_uri(file_uri=GCS_URI, mime_type=detect_mime(GCS_URI))
        result = call_gemini(part)
        print(f"[성공] 응답 시간={result['elapsed_sec']}s, 토큰={result['input_tokens']}")
        print_result("GCS URI 직접 전달", result)
    except Exception as e:
        print(f"[실패] {e}")

---
## Phase 2 — base64 vs GCS URI 속도 비교

**배경**: HTTP URL 직접 전달은 불안정하므로 제외.  
실제 운영 가능한 두 방식만 비교한다.

| 방식 | 이미지 소스 | 측정 구간 |
|------|------------|----------|
| A: base64 (현재) | DOWNLOAD_URL 다운로드 → bytes | 다운로드 + API 응답 |
| B: GCS URI | GCS_URI 직접 전달 | API 응답만 |

> **공정 비교 조건**: A와 B는 동일한 이미지여야 한다.  
> DOWNLOAD_URL의 이미지 = GCS_URI의 이미지 (포맷 차이는 있어도 됨)

In [6]:
N_REPEAT = 5

print(f"반복 횟수: {N_REPEAT}회")
print(f"DOWNLOAD_URL: {'설정됨' if DOWNLOAD_URL else '미설정 — 방식 A 건너뜀'}")
print(f"GCS_URI     : {'설정됨' if GCS_URI     else '미설정 — 방식 B 건너뜀'}")

반복 횟수: 5회
DOWNLOAD_URL: 설정됨
GCS_URI     : 설정됨


In [7]:
# 방식 A: base64 (현재 방식)
# DOWNLOAD_URL에서 다운로드 후 bytes로 전달
results_a = []

if not DOWNLOAD_URL:
    print("DOWNLOAD_URL 미설정 — 방식 A 건너뜀")
else:
    print(f"[방식 A] base64 — {N_REPEAT}회 측정")
    for i in range(N_REPEAT):
        t_start   = time.perf_counter()
        img_bytes = download_image(DOWNLOAD_URL)
        t_dl      = round(time.perf_counter() - t_start, 3)

        part   = types.Part.from_bytes(data=img_bytes, mime_type=detect_mime(DOWNLOAD_URL))
        result = call_gemini(part)
        result["download_sec"] = t_dl
        result["total_sec"]    = round(t_dl + result["elapsed_sec"], 3)
        results_a.append(result)
        print(f"  [{i+1}/{N_REPEAT}] 전체={result['total_sec']}s "
              f"(DL={t_dl}s + API={result['elapsed_sec']}s) "
              f"토큰={result['input_tokens']}")

    avg_a = sum(r['total_sec'] for r in results_a) / N_REPEAT
    print(f"  -> 평균 총 시간: {avg_a:.3f}s")

[방식 A] base64 — 5회 측정
  [1/5] 전체=2.522s (DL=0.177s + API=2.345s) 토큰=1312
  [2/5] 전체=2.063s (DL=0.093s + API=1.97s) 토큰=1312
  [3/5] 전체=2.625s (DL=0.132s + API=2.493s) 토큰=1312
  [4/5] 전체=2.107s (DL=0.116s + API=1.991s) 토큰=1312
  [5/5] 전체=2.087s (DL=0.11s + API=1.977s) 토큰=1312
  -> 평균 총 시간: 2.281s


In [8]:
# 방식 B: GCS URI (gs://) 직접 전달
results_b = []

if not GCS_URI:
    print("GCS_URI 미설정 — 방식 B 건너뜀")
else:
    print(f"[방식 B] GCS URI — {N_REPEAT}회 측정")
    for i in range(N_REPEAT):
        try:
            part   = types.Part.from_uri(file_uri=GCS_URI, mime_type=detect_mime(GCS_URI))
            result = call_gemini(part)
            result["download_sec"] = 0.0
            result["total_sec"]    = result["elapsed_sec"]
            results_b.append(result)
            print(f"  [{i+1}/{N_REPEAT}] 전체={result['total_sec']}s (API만) "
                  f"토큰={result['input_tokens']}")
        except Exception as e:
            print(f"  [{i+1}/{N_REPEAT}] 실패: {e}")
            break

    if results_b:
        avg_b = sum(r['total_sec'] for r in results_b) / len(results_b)
        print(f"  -> 평균 총 시간: {avg_b:.3f}s")

[방식 B] GCS URI — 5회 측정
  [1/5] 전체=1.682s (API만) 토큰=1312
  [2/5] 전체=1.469s (API만) 토큰=1312
  [3/5] 전체=1.285s (API만) 토큰=1312
  [4/5] 전체=1.447s (API만) 토큰=1312
  [5/5] 전체=1.459s (API만) 토큰=1312
  -> 평균 총 시간: 1.468s


In [9]:
# Phase 2 결과 요약
rows = []
if results_a:
    n = len(results_a)
    rows.append(("A: base64 (현재)",
                 sum(r['total_sec']    for r in results_a) / n,
                 sum(r['elapsed_sec']  for r in results_a) / n,
                 sum(r['download_sec'] for r in results_a) / n,
                 sum(r['input_tokens'] for r in results_a) / n))
if results_b:
    n = len(results_b)
    rows.append(("B: GCS URI",
                 sum(r['total_sec']    for r in results_b) / n,
                 sum(r['elapsed_sec']  for r in results_b) / n,
                 0.0,
                 sum(r['input_tokens'] for r in results_b) / n))

if not rows:
    print("실행된 테스트 없음. URL 설정 후 재실행하세요.")
else:
    print("=" * 65)
    print("Phase 2 결과")
    print("=" * 65)
    print(f"{'방식':<22} {'총 시간':>9} {'API 시간':>10} {'DL 시간':>9} {'입력 토큰':>10}")
    print("-" * 60)
    for label, total, api, dl, tok in rows:
        print(f"{label:<22} {total:>8.3f}s {api:>9.3f}s {dl:>8.3f}s {tok:>10.0f}")

    if len(rows) == 2:
        base = rows[0][1]
        diff = rows[1]
        pct  = (base - diff[1]) / base * 100
        direction = "단축" if pct > 0 else "증가"
        print(f"\n{diff[0]}: 기준(base64) 대비 총 시간 {direction} {abs(pct):.1f}%")
        # DL 시간만큼 순수 절감
        dl_saving = rows[0][3]
        print(f"다운로드 단계 제거로 절감되는 시간: {dl_saving:.3f}s/건")
        print(f"4만 건/시간 기준 절감: {dl_saving * 40000 / 3600:.1f}시간 상당의 처리 시간 단축")

Phase 2 결과
방식                          총 시간     API 시간     DL 시간      입력 토큰
------------------------------------------------------------
A: base64 (현재)            2.281s     2.155s    0.126s       1312
B: GCS URI                1.468s     1.468s    0.000s       1312

B: GCS URI: 기준(base64) 대비 총 시간 단축 35.6%
다운로드 단계 제거로 절감되는 시간: 0.126s/건
4만 건/시간 기준 절감: 1.4시간 상당의 처리 시간 단축


---
## Phase 3 — `media_resolution` 파라미터 최적화

**핵심**: 이미지당 할당 토큰 수를 직접 제어하는 파라미터

| 설정값 | 이미지 토큰 | 비용 비율 | 권장 케이스 |
|--------|------------|----------|------------|
| `MEDIA_RESOLUTION_HIGH` (기본값) | ~1,120 | 4x | 작은 텍스트 읽기, 세밀한 분석 |
| `MEDIA_RESOLUTION_MEDIUM` | ~560 | 2x | 일반 캡셔닝 |
| `MEDIA_RESOLUTION_LOW` | ~280 | 1x | 전체 구도, 색상 파악 |

> **주의**: `'low'` 가 아닌 `'MEDIA_RESOLUTION_LOW'` 전체 이름을 사용해야 한다. (Vertex AI 프로토 enum)  
> `countTokens` API는 이 파라미터를 반영하지 않으므로 `generate_content` 결과로만 확인 가능.

In [ ]:
PHASE3_URL = GCS_URI  # gs:// 가 가장 안정적
resolution_results = {}

RESOLUTIONS = [
    ("LOW",    "MEDIA_RESOLUTION_LOW"),
    ("MEDIUM", "MEDIA_RESOLUTION_MEDIUM"),
    ("HIGH",   "MEDIA_RESOLUTION_HIGH"),
]

if not PHASE3_URL:
    print("GCS_URI 미설정 — Phase 3 건너뜀")
else:
    print(f"URL: {PHASE3_URL}\n")
    for label, res_value in RESOLUTIONS:
        print(f"[{label}] 테스트 중...")
        try:
            part   = types.Part.from_uri(file_uri=PHASE3_URL, mime_type=detect_mime(PHASE3_URL))
            result = call_gemini(part, media_resolution=res_value)
            resolution_results[label] = result
            print(f"  토큰: {result['input_tokens']} | 시간: {result['elapsed_sec']}s")
            print(f"  캡션: {result['text'][:150]}")
            print()
        except Exception as e:
            print(f"  실패: {e}\n")

In [11]:
# Phase 3 결과 요약
if not resolution_results:
    print("실행된 결과 없음")
else:
    print("=" * 55)
    print("Phase 3 — media_resolution 비교 결과")
    print("=" * 55)
    print(f"{'해상도':<10} {'입력 토큰':>10} {'응답 시간':>10} {'HIGH 대비 절감':>14}")
    print("-" * 44)

    base_tok = resolution_results.get("HIGH", {}).get("input_tokens") or 1
    for label in ["HIGH", "MEDIUM", "LOW"]:
        if label not in resolution_results:
            continue
        r      = resolution_results[label]
        saving = (base_tok - r['input_tokens']) / base_tok * 100
        print(f"{label:<10} {r['input_tokens']:>10} {r['elapsed_sec']:>9.3f}s {saving:>13.1f}%")

    # 월 비용 추정
    PRICE_PER_M    = 0.10  # gemini-2.5-flash-lite: $0.10/M input tokens
    MONTHLY_VOLUME = 40_000 * 24 * 30
    print(f"\n4만 건/시간 x 30일 = {MONTHLY_VOLUME:,}건/월 기준 월 비용 추정:")
    for label in ["HIGH", "MEDIUM", "LOW"]:
        if label not in resolution_results:
            continue
        tok  = resolution_results[label]['input_tokens']
        cost = tok * MONTHLY_VOLUME / 1_000_000 * PRICE_PER_M
        print(f"  {label:8s}: ${cost:,.2f} USD/월")

Phase 3 — media_resolution 비교 결과
해상도             입력 토큰      응답 시간     HIGH 대비 절감
--------------------------------------------
HIGH             1312     1.533s           0.0%
MEDIUM            280     1.048s          78.7%
LOW                88     1.685s          93.3%

4만 건/시간 x 30일 = 28,800,000건/월 기준 월 비용 추정:
  HIGH    : $3,778.56 USD/월
  MEDIUM  : $806.40 USD/월
  LOW     : $253.44 USD/월


---
## Phase 4 — 이미지 리사이즈 최적화

**배경**: Gemini 토큰 계산 규칙
- 양쪽 모두 <= **384px** → **258 토큰 고정** (타일 분할 없음)
- 그 이상 → 768×768px 타일로 분할, 타일당 258 토큰

현재 CDN 변환 파이프라인은 **480px** 기준 리사이즈 중.  
480 > 384 이므로 타일 분할이 발생하여 토큰이 증가한다.

**실험**: 동일 이미지를 480px / 384px / 300px로 리사이즈 후 실제 소비 토큰 비교

In [ ]:
def resize_image_webp(img_bytes: bytes, max_width: int, quality: int = 75) -> bytes:
    img = Image.open(io.BytesIO(img_bytes))
    w, h = img.size
    if w > max_width:
        new_h = int(h * max_width / w)
        img   = img.resize((max_width, new_h), Image.LANCZOS)
    out = io.BytesIO()
    img.save(out, format="WEBP", quality=quality)
    return out.getvalue()


# DOWNLOAD_URL로 원본 다운로드 후 리사이즈 비교
PHASE4_URL     = DOWNLOAD_URL
resize_results = {}
sizes          = {}

if not PHASE4_URL:
    print("DOWNLOAD_URL 미설정 — Phase 4 건너뜀")
else:
    print(f"원본 다운로드: {PHASE4_URL[:60]}")
    original_bytes = download_image(PHASE4_URL)
    orig_img       = Image.open(io.BytesIO(original_bytes))
    print(f"원본 크기: {orig_img.size} ({len(original_bytes)/1024:.1f} KB)\n")

    sizes = {
        "원본":              original_bytes,
        "480px (현재 CDN)": resize_image_webp(original_bytes, 480),
        "384px (제안)":      resize_image_webp(original_bytes, 384),
        "300px":             resize_image_webp(original_bytes, 300),
    }

    for label, img_bytes in sizes.items():
        img_info = Image.open(io.BytesIO(img_bytes))
        part     = types.Part.from_bytes(data=img_bytes, mime_type="image/webp")
        result   = call_gemini(part)
        resize_results[label] = {
            "size":    img_info.size,
            "file_kb": len(img_bytes) / 1024,
            "tokens":  result['input_tokens'],
        }
        print(f"[{label}] {img_info.size} / {len(img_bytes)/1024:.1f}KB -> 토큰: {result['input_tokens']}")

In [13]:
# 384px 경계값 테스트 (토큰 기준이 정확히 384px인지 전후 확인)
if not PHASE4_URL:
    print("건너뜀")
else:
    print("경계값 테스트 (384px 기준 전후):\n")
    for w in [390, 385, 384, 383, 380]:
        img_bytes = resize_image_webp(original_bytes, max_width=w)
        img_info  = Image.open(io.BytesIO(img_bytes))
        part      = types.Part.from_bytes(data=img_bytes, mime_type="image/webp")
        result    = call_gemini(part)
        print(f"  {w}px -> 실제={img_info.size}, 토큰={result['input_tokens']}")

경계값 테스트 (384px 기준 전후):

  390px -> 실제=(390, 390), 토큰=1312
  385px -> 실제=(385, 385), 토큰=1312
  384px -> 실제=(384, 384), 토큰=280
  383px -> 실제=(383, 383), 토큰=280
  380px -> 실제=(380, 380), 토큰=280


In [14]:
# 캡셔닝 품질 비교 (480px vs 384px)
if not PHASE4_URL or not sizes:
    print("건너뜀")
else:
    print("캡셔닝 품질 비교 (480px vs 384px):\n")
    for label in ["480px (현재 CDN)", "384px (제안)"]:
        if label not in sizes:
            continue
        part   = types.Part.from_bytes(data=sizes[label], mime_type="image/webp")
        result = call_gemini(part)
        print(f"[{label}]")
        print(f"  토큰: {result['input_tokens']} | 시간: {result['elapsed_sec']}s")
        print(f"  캡션: {result['text']}")
        print()

캡셔닝 품질 비교 (480px vs 384px):

[480px (현재 CDN)]
  토큰: 1312 | 시간: 1.217s
  캡션: 세련된 데님 셔츠와 스커트 세트로 시크한 매력을 완성하세요. 편안하면서도 스타일리시한 데일리룩을 연출할 수 있습니다.

[384px (제안)]
  토큰: 280 | 시간: 1.386s
  캡션: ## 마케팅용 한국어 캡션 (2문장)

1. **청량한 데님 소재와 세련된 디자인으로 어떤 스타일에도 멋스럽게 연출 가능한 만능 아이템!**
2. **편안함과 스타일을 동시에 잡은 데님 셔츠로 당신의 일상에 특별함을 더해보세요.**



In [15]:
# Phase 4 결과 요약
if not resize_results:
    print("실행된 결과 없음")
else:
    print("=" * 60)
    print("Phase 4 — 이미지 리사이즈 최적화 결과")
    print("=" * 60)
    print(f"{'크기':>18} {'해상도':>12} {'파일 크기':>10} {'토큰 수':>8} {'절감율':>8}")
    print("-" * 58)

    base_tok = list(resize_results.values())[0]['tokens'] or 1
    for label, r in resize_results.items():
        saving = (base_tok - r['tokens']) / base_tok * 100
        print(f"{label:>18} {str(r['size']):>12} {r['file_kb']:>8.1f}KB "
              f"{r['tokens']:>8} {saving:>7.1f}%")

    k480, k384 = "480px (현재 CDN)", "384px (제안)"
    if k480 in resize_results and k384 in resize_results:
        diff = resize_results[k480]['tokens'] - resize_results[k384]['tokens']
        pct  = diff / resize_results[k480]['tokens'] * 100
        print(f"\n480px -> 384px 변경 시: 토큰 {diff}개 절감 ({pct:.1f}%)")

    PRICE_PER_M    = 0.10
    MONTHLY_VOLUME = 40_000 * 24 * 30
    print(f"\n4만 건/시간 x 30일 = {MONTHLY_VOLUME:,}건/월 기준 월 비용 추정:")
    for label, r in resize_results.items():
        cost = r['tokens'] * MONTHLY_VOLUME / 1_000_000 * PRICE_PER_M
        print(f"  {label:>18}: ${cost:,.2f} USD/월")

Phase 4 — 이미지 리사이즈 최적화 결과
                크기          해상도      파일 크기     토큰 수      절감율
----------------------------------------------------------
                원본   (800, 800)    135.0KB     1312     0.0%
    480px (현재 CDN)   (480, 480)     14.9KB     1312     0.0%
        384px (제안)   (384, 384)     10.3KB      280    78.7%
             300px   (300, 300)      6.8KB      280    78.7%

480px -> 384px 변경 시: 토큰 1032개 절감 (78.7%)

4만 건/시간 x 30일 = 28,800,000건/월 기준 월 비용 추정:
                  원본: $3,778.56 USD/월
      480px (현재 CDN): $3,778.56 USD/월
          384px (제안): $806.40 USD/월
               300px: $806.40 USD/월


---
## 최종 결과 종합 및 권장사항

In [16]:
print("=" * 65)
print("Gemini 이미지 캡셔닝 최적화 실험 — 종합 결과")
print("=" * 65)

print("""
[Phase 1] URL 접근 가능성 — 확정 결론
  외부 CDN URL (한섬/쿠팡): 불안정, 운영 불가
  GCS CDN HTTP URL (IP)   : 실패
  GCS URI (gs://)         : 성공 — 유일하게 신뢰 가능
""")

print("[Phase 2] base64 vs GCS URI 속도 비교")
if rows:
    for label, total, api, dl, tok in rows:
        print(f"  {label:<22}: 총 {total:.3f}s (DL={dl:.3f}s + API={api:.3f}s) | 토큰 {tok:.0f}")
    if len(rows) == 2:
        pct = (rows[0][1] - rows[1][1]) / rows[0][1] * 100
        print(f"  GCS URI 방식: 총 처리 시간 {abs(pct):.1f}% 단축")
else:
    print("  -> URL 설정 후 재실행 필요")

print("\n[Phase 3] media_resolution 토큰 절감")
if resolution_results:
    base = resolution_results.get('HIGH', {}).get('input_tokens') or 1
    for label in ['HIGH', 'MEDIUM', 'LOW']:
        if label in resolution_results:
            r = resolution_results[label]
            s = (base - r['input_tokens']) / base * 100
            print(f"  {label:8s}: {r['input_tokens']} 토큰 | {r['elapsed_sec']}s | 절감 {s:.1f}%")
else:
    print("  -> URL 설정 후 재실행 필요")

print("\n[Phase 4] 이미지 리사이즈 최적화")
if resize_results:
    base = list(resize_results.values())[0]['tokens'] or 1
    for label, r in resize_results.items():
        s = (base - r['tokens']) / base * 100
        print(f"  {label:>18}: {r['tokens']} 토큰 ({r['file_kb']:.1f}KB) | 절감 {s:.1f}%")
else:
    print("  -> URL 설정 후 재실행 필요")

print()
print("-" * 65)
print("""권장 최적화 적용 순서

  1. [즉시, 임팩트 최대] CDN 리사이즈 기준 480px -> 384px 이하로 변경
     코드 한 줄 수정으로 토큰 약 -78% 달성 (1312 -> 280)
     aid-gemi-preprocessing 파이프라인의 max_width 값만 변경하면 됨

  2. [즉시] 캡셔닝 파이프라인에서 base64 대신 GCS URI (gs://) 사용
     다운로드/인코딩 단계 제거로 처리 시간 단축
     CDN 파이프라인이 이미 GCS에 업로드하므로 gs:// 경로 그대로 활용 가능

  3. [Phase 3 결과 확인 후] media_resolution 조정 검토
     384px 리사이즈로 이미 LOW 수준 토큰이 되므로 추가 효과 제한적
     단, media_resolution=MEDIA_RESOLUTION_MEDIUM 으로 추가 절감 가능 여부 확인

  4. [중장기] Gemini Batch Prediction API 검토
     실시간 불필요 시 비용 50% 절감 가능
     GCS URI 방식과 자연스럽게 연동
""")

Gemini 이미지 캡셔닝 최적화 실험 — 종합 결과

[Phase 1] URL 접근 가능성 — 확정 결론
  외부 CDN URL (한섬/쿠팡): 불안정, 운영 불가
  GCS CDN HTTP URL (IP)   : 실패
  GCS URI (gs://)         : 성공 — 유일하게 신뢰 가능

[Phase 2] base64 vs GCS URI 속도 비교
  A: base64 (현재)        : 총 2.281s (DL=0.126s + API=2.155s) | 토큰 1312
  B: GCS URI            : 총 1.468s (DL=0.000s + API=1.468s) | 토큰 1312
  GCS URI 방식: 총 처리 시간 35.6% 단축

[Phase 3] media_resolution 토큰 절감
  HIGH    : 1312 토큰 | 1.533s | 절감 0.0%
  MEDIUM  : 280 토큰 | 1.048s | 절감 78.7%
  LOW     : 88 토큰 | 1.685s | 절감 93.3%

[Phase 4] 이미지 리사이즈 최적화
                  원본: 1312 토큰 (135.0KB) | 절감 0.0%
      480px (현재 CDN): 1312 토큰 (14.9KB) | 절감 0.0%
          384px (제안): 280 토큰 (10.3KB) | 절감 78.7%
               300px: 280 토큰 (6.8KB) | 절감 78.7%

-----------------------------------------------------------------
권장 최적화 적용 순서

  1. [즉시, 임팩트 최대] CDN 리사이즈 기준 480px -> 384px 이하로 변경
     코드 한 줄 수정으로 토큰 약 -78% 달성 (1312 -> 280)
     aid-gemi-preprocessing 파이프라인의 max_width 값만 변경하면 됨

  2. [즉시] 캡셔닝 파이프라인에서 b

## Phase 5 — 대량 처리(4만 건/시간)를 위한 비동기(Async) 속도 최적화
- **배경**: 목표 처리량인 4만 건/시간(초당 약 11건)을 달성하려면 동기식(순차) 처리로는 한계가 있습니다.
- **실험**: 동일한 이미지 5건을 요청할 때, 동기식(Sync)과 비동기식(Async) 클라이언트(`client.aio`)의 총 소요 시간을 비교합니다.

In [17]:
import asyncio

# Quota 초과(HTTP 429) 방지를 위한 동시성 제어 (한 번에 최대 10개까지만 요청)
semaphore = asyncio.Semaphore(10)

async def call_gemini_async(uri: str, prompt: str):
    async with semaphore:
        part = types.Part.from_uri(file_uri=uri, mime_type="image/webp")
        # 비동기 클라이언트(client.aio) 사용
        resp = await client.aio.models.generate_content(
            model=MODEL,
            contents=[part, types.Part.from_text(text=prompt)],
            config=types.GenerateContentConfig(temperature=0),
        )
        return resp

async def run_async_test(uris: list):
    print(f"[비동기] {len(uris)}건 동시 처리 시작...")
    t0 = time.perf_counter()
    
    # 5개의 요청을 동시에 발송
    tasks = [call_gemini_async(uri, CAPTION_PROMPT) for uri in uris]
    results = await asyncio.gather(*tasks)
    
    elapsed = time.perf_counter() - t0
    print(f"[비동기] 완료! 총 소요 시간: {elapsed:.3f}s (평균 1건당 {elapsed/len(uris):.3f}s)")
    return elapsed

def run_sync_test(uris: list):
    print(f"[동기] {len(uris)}건 순차 처리 시작...")
    t0 = time.perf_counter()
    
    # 5개의 요청을 하나씩 순서대로 발송 (현재 로직)
    for uri in uris:
        part = types.Part.from_uri(file_uri=uri, mime_type="image/webp")
        client.models.generate_content(
            model=MODEL,
            contents=[part, types.Part.from_text(text=CAPTION_PROMPT)],
            config=types.GenerateContentConfig(temperature=0),
        )
        
    elapsed = time.perf_counter() - t0
    print(f"[동기] 완료! 총 소요 시간: {elapsed:.3f}s (평균 1건당 {elapsed/len(uris):.3f}s)")
    return elapsed

# -----------------------------------------------------------------
# 테스트 실행 (Jupyter 환경에서는 최상위에서 await을 바로 사용할 수 있습니다)
if not GCS_URI:
    print("GCS_URI 미설정 — Phase 5 건너뜀")
else:
    TEST_COUNT = 5
    test_uris = [GCS_URI] * TEST_COUNT
    
    print("=" * 60)
    print("Phase 5 — 동기(Sync) vs 비동기(Async) 속도 비교")
    print("=" * 60)
    
    sync_time = run_sync_test(test_uris)
    print("-" * 60)
    async_time = await run_async_test(test_uris)
    print("=" * 60)
    
    speed_up = sync_time / async_time
    print(f"💡 결론: 비동기 처리가 동기 처리 대비 약 {speed_up:.1f}배 빠름!")
    print("💡 4만 건/시간(초당 11건) 처리를 위해서는 client.aio 도입이 필수적입니다.")

Phase 5 — 동기(Sync) vs 비동기(Async) 속도 비교
[동기] 5건 순차 처리 시작...
[동기] 완료! 총 소요 시간: 11.112s (평균 1건당 2.222s)
------------------------------------------------------------
[비동기] 5건 동시 처리 시작...
[비동기] 완료! 총 소요 시간: 3.053s (평균 1건당 0.611s)
💡 결론: 비동기 처리가 동기 처리 대비 약 3.6배 빠름!
💡 4만 건/시간(초당 11건) 처리를 위해서는 client.aio 도입이 필수적입니다.


---
## 실험 결과 요약

### Phase 1 — URL 접근 가능성
| 방식 | 결과 |
|------|------|
| 외부 CDN HTTP URL | 간헐적 성공, 운영 불가 |
| GCS CDN HTTP URL (IP) | 실패 |
| GCS URI (`gs://`) | **성공** — 유일하게 신뢰 가능 |

---

### Phase 2 — base64 vs GCS URI 속도 (5회 평균)
| 방식 | 총 시간 | API 시간 | 다운로드 |
|------|--------|---------|---------|
| base64 (현재) | 2.281s | 2.155s | 0.126s |
| GCS URI | **1.468s** | 1.468s | 0s |

> GCS URI 전환 시 총 처리 시간 **35.6% 단축**. 4만 건/시간 기준 약 **1.4시간 상당** 절감.

---

### Phase 3 — `media_resolution` 토큰 절감
| 설정 | 입력 토큰 | HIGH 대비 절감 | 월 비용 (2,880만 건) |
|------|----------|--------------|-------------------|
| HIGH (기본값) | 1,312 | — | $3,779 |
| MEDIUM | 280 | **78.7%** | $806 |
| LOW | 88 | **93.3%** | $253 |

> 캡셔닝 품질은 MEDIUM으로도 충분. LOW는 전체 구도·색상 파악 수준.

---

### Phase 4 — 이미지 리사이즈 (384px 임계값)
| 크기 | 토큰 수 | 절감율 |
|------|--------|-------|
| 원본 (800px) | 1,312 | — |
| 480px (현재 CDN) | 1,312 | 0% |
| **384px (권장)** | **280** | **78.7%** |
| 300px | 280 | 78.7% |

> 385px부터 1,312 토큰, 384px 이하는 280 토큰으로 급감. CDN 리사이즈 기준을 **384px 이하**로 변경 권장.

---

### Phase 5 — 동기 vs 비동기 처리 (5건 기준)
| 방식 | 총 시간 | 건당 시간 |
|------|--------|---------|
| Sync (순차) | 11.1s | 2.22s |
| Async (`client.aio`) | **3.1s** | **0.61s** |

> 비동기 처리가 **3.6배 빠름**. 4만 건/시간(초당 11건) 달성을 위해 `client.aio` + Semaphore 필수.

---

### 권장 운영 설정

| 항목 | 설정 |
|------|------|
| 이미지 전달 | GCS URI (`gs://`) |
| 해상도 | `MEDIA_RESOLUTION_MEDIUM` |
| 리사이즈 | 384px 이하 |
| 호출 방식 | `client.aio` + `asyncio.Semaphore` |